
# RAG — plateforme B2B d'arômes et de fragrances (équipe 10)

Ce notebook déroule toute la démo sur Colab :

1. installation (dépôt, Python, Ollama, modèles) ;
2. génération du dataset et indexation ;
3. *(facultatif, ~15 min)* évaluation du modèle retenu (`qwen3-embedding:4b` + `qwen2.5:3b`) sur les 30 cas de référence et les 10 cas inédits, sur GPU, et comparaison V1 / V2 ;
4. lancement de l'API (V2.2), parcours de démonstration et résultats mesurés ;
5. exposition publique via ngrok (lien Swagger à transmettre aux interfaces).

**Avant de lancer** : choisir un environnement d'exécution avec GPU (*Exécution → Modifier le type d'exécution → GPU T4*). Ajouter ton token ngrok dans les *Secrets* Colab (icône 🔑 à gauche) sous le nom `NGROK_AUTHTOKEN`, avec « Accès au notebook » activé. Facultatif : ton domaine statique ngrok (`xxx.ngrok-free.app` ou `.dev`) sous le nom `NGROK_DOMAIN`, pour garder le même lien à chaque relance. Ne jamais écrire un token en clair dans une cellule.

**Démo rapide** : exécuter les sections 1, 2, 4 et 5 (≈ 10 min) en sautant la section 3.

**Après une déconnexion de Colab**, tout est perdu (modèles, index, API, tunnel) : relancer les sections 1, 2, 4 et 5 dans l'ordre. Dépannage en fin de notebook et dans `docs/REPRODUCTION.md`.

## 1. Installation

In [ ]:
import os, time
BRANCH = "main"
%cd /content
if not os.path.exists("/content/equipe10"):
    !git clone -b $BRANCH https://github.com/Hetic-Hackathon-Givaudan-2026-CHARK/equipe10.git
%cd /content/equipe10
# Les avertissements « pip's dependency resolver… » concernent des paquets préinstallés de Colab : sans effet.
!pip install -q -r requirements.txt pyngrok

In [ ]:
# Ollama (serveur de modèles local). « Unable to detect NVIDIA/AMD GPU » est normal sur Colab :
# vérifier le GPU avec !nvidia-smi (Tesla T4 attendu).
!apt-get install -y -qq zstd > /dev/null
!which ollama > /dev/null || (curl -fsSL https://ollama.com/install.sh | sh)
!pgrep -f "[o]llama serve" > /dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
time.sleep(5)
!curl -s http://127.0.0.1:11434/api/tags

In [ ]:
!ollama pull qwen3-embedding:4b
!ollama pull qwen2.5:3b
!ollama list

## 2. Dataset et indexation

In [ ]:
%env EMBED_MODEL=qwen3-embedding:4b
%env LLM_MODEL=qwen2.5:3b
!python generate_dataset.py
!head -c 1500 givaudan_data.txt

In [ ]:
# Tests automatiques (données, ingestion, API, reclassement, comparaison V1/V2) — sans Ollama
!python -m pytest -q

In [ ]:
# Indexation dans ChromaDB avec qwen3-embedding:4b (obligatoire pour /search et /ask)
!python clean.py && python ingest.py

## 3. Évaluation (facultatif)

Protocole et seuils : `docs/EVALUATION.md` ; comparaison V1 / V2 : `docs/COMPARAISON_V1_V2.md`. Il faut réindexer (section 2) à chaque changement de modèle d'embeddings.

In [ ]:
# Baseline lexicale sans IA (TF-IDF), avec et sans reclassement : ne nécessite pas Ollama
!python baseline_tfidf.py --label baseline_tfidf | tail -6
!python baseline_tfidf.py --rerank --label baseline_tfidf_rerank | tail -6
!python baseline_tfidf.py --rerank --cases data/eval/unseen_cases.json --label baseline_tfidf_rerank_inedits | tail -6

In [ ]:
# Modèle publié (V2.2) sur GPU : exactitude, refus, latence
!python evaluate.py --label v22_qwen3_llm3b_gpu
!python evaluate.py --label v22_qwen3_llm3b_gpu_inedits --cases data/eval/unseen_cases.json

In [ ]:
# Comparaison V1 / V2 sur les mêmes cas (sans IA ; ajouter --models pour les vrais modèles, nécessite qwen2.5:1.5b)
!python compare_v1_v2.py | head -30

In [ ]:
# Synthèse de toutes les configurations (baseline, nomic, qwen3, cas inédits, GPU)
!python report.py

In [ ]:
# Télécharger tous les rapports d'évaluation (à pousser dans results/ sur GitHub)
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

## 4. API (V2.2), parcours de démonstration et résultats

In [ ]:
# Lance (ou relance) l'API en arrière-plan et attend qu'elle réponde.
%cd /content/equipe10
!pgrep -f "[o]llama serve" > /dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
!pkill -f "[a]pi.py"
!nohup python api.py > /tmp/fastapi.log 2>&1 &
import requests
for _ in range(30):
    try:
        health = requests.get("http://127.0.0.1:8000/health", timeout=2).json()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("L'API ne démarre pas : voir !tail -30 /tmp/fastapi.log")
print(health)
assert health["indexed_documents"] > 0, "Index vide : relancer la section 2 (clean.py && ingest.py)"

In [ ]:
# Parcours de démonstration (voir docs/DEMO.md) : brief -> proposition -> suivi -> garde-fou
API = "http://127.0.0.1:8000"
parcours = [
    ("Brief", "Je cherche un arôme mangue pour une boisson protéinée, plutôt naturel et vegan. Que proposez-vous ?", "dairy_aroma"),
    ("Proposition", "Quel arôme utiliser pour masquer l'amertume des protéines dans un shake ?", "dairy_aroma"),
    ("Échantillons", "Pourquoi la V1 de l'échantillon de boisson protéinée de NutriPeak n'a-t-elle pas été validée ?", "all"),
    ("Suivi", "Quel est le statut de la commande ORD-2026-007 ?", "all"),
    ("Garde-fou", "Donne-moi la formule exacte de FRG-101 avec le pourcentage de chaque ingrédient.", "all"),
]
for etape, question, domaine in parcours:
    start = time.time()
    body = requests.post(f"{API}/ask", json={"question": question, "domain": domaine}, timeout=120).json()
    print(f"### {etape} ({time.time() - start:.1f} s, source : {body.get('answer_source')})")
    print("Q :", question)
    print("R :", body.get("answer", body), "\n")

In [ ]:
# Résultats mesurés, tels que l'interface de démo peut les afficher (GET /evaluation)
ev = requests.get(f"{API}/evaluation").json()
ref, ined, comp = ev["cas_de_reference"], ev["cas_inedits"], ev["comparaison_v1_v2"]
print(f"Modèle : {ev['modele']['embeddings']} + {ev['modele']['llm']} ({ev['version']})")
print(f"30 cas de référence : bon document top 4 {ref['bon_document_top4']:.0%}, réponses exactes {ref['reponses_exactes']:.0%}, refus corrects {ref['refus_corrects']:.0%}")
print(f"10 cas inédits      : bon document top 4 {ined['bon_document_top4']:.0%}, réponses exactes {ined['reponses_exactes']:.0%}, refus corrects {ined['refus_corrects']:.0%}")
if comp:
    print(f"V1 -> V2 : documents {comp['documents_indexes']['v1']} -> {comp['documents_indexes']['v2']}, "
          f"cas couverts par les données {comp['couverture_30_cas']['v1']:.0%} -> {comp['couverture_30_cas']['v2']:.0%}")
    if comp["reponses_exactes"]["v1"] is not None:
        print(f"Réponses exactes V1 -> V2 : {comp['reponses_exactes']['v1']:.0%} -> {comp['reponses_exactes']['v2']:.0%}")

## 5. Lien public (ngrok)

Le lien Swagger affiché est celui à transmettre au tech lead. Il ne fonctionne que tant que cette session Colab est active.

In [ ]:
from pyngrok import ngrok, conf

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    from getpass import getpass
    token = getpass("Token ngrok : ")
try:
    from google.colab import userdata
    domain = userdata.get("NGROK_DOMAIN")
except Exception:
    domain = None

# Passer le token à la configuration (évite l'erreur « Authtoken already set » d'un second appel).
conf.get_default().auth_token = token
ngrok.kill()
# 127.0.0.1 explicite : « localhost » peut être résolu en IPv6 alors que l'API écoute en IPv4 (ERR_NGROK_8012).
if not domain:
    options = {}
elif conf.get_default().config_version == "3":
    options = {"url": f"https://{domain}"}
else:
    options = {"domain": domain}
public_url = ngrok.connect("http://127.0.0.1:8000", **options).public_url
print("Swagger    :", f"{public_url}/docs")
print("Ping       :", f"{public_url}/demo/ping")
print("Résultats  :", f"{public_url}/evaluation")

## Dépannage

| Symptôme | Cause | Solution |
|---|---|---|
| `ERR_NGROK_3200` (endpoint offline) | session Colab arrêtée ou tunnel fermé | relancer les sections 1, 2, 4 et 5 |
| `ERR_NGROK_8012` (connection refused) | l'API ne tourne pas | relancer la cellule de lancement de l'API (section 4), puis la section 5 |
| `/search` ou `/ask` renvoient 503 « base vectorielle vide » | indexation non faite | relancer la dernière cellule de la section 2 |
| `SecretNotFoundError: NGROK_AUTHTOKEN` | secret absent | ajouter le secret (icône 🔑), activer « Accès au notebook » |
| page « Visit Site » de ngrok | avertissement du plan gratuit | cliquer, ou en-tête `ngrok-skip-browser-warning: true` depuis une interface |
| réponses très lentes (> 20 s) | pas de GPU | *Exécution → Modifier le type d'exécution → GPU T4*, puis tout relancer |

Colab gratuit coupe après ~90 min d'inactivité et après quelques heures au maximum : garder l'onglet ouvert et actif pendant la démo.